In [1]:
import pandas as pd
import numpy as np
from keras.models import load_model
import pickle

In [2]:
# loading the model , encoder and scaler
model = load_model('model.keras')

# encoder
with open('one_hot_encoder.pkl','rb') as file:
    one_hot_enc_obj = pickle.load(file)

with open('lable_encoder.pkl','rb') as file:
    lable_encoder_obj = pickle.load(file)

# scaler
with open('scaler.pkl','rb') as file:
    scaler_obj = pickle.load(file)

In [3]:
# Example input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}

input_df = pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [4]:
# converting geography into numerical data
geo_array = one_hot_enc_obj.transform([[input_data['Geography']]]).toarray()
geo_df = pd.DataFrame(data=geo_array, columns=one_hot_enc_obj.get_feature_names_out())
geo_df

e:\ML_projects\Chrun Modelling\venv\Lib\site-packages\sklearn\utils\validation.py:2830: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [5]:
# combinig the geo data and input data
input_df = pd.concat([input_df.reset_index(drop=True),geo_df],axis=1)
input_df = input_df.drop(columns=['Geography']) # removing geography columns

In [6]:
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [7]:
# using lable encoder for gender
input_df['Gender'] = lable_encoder_obj.transform(input_df['Gender'])
input_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [8]:
# Scaling the data
scaled_array = scaler_obj.transform(input_df)
scaled_array

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

In [9]:
# prduction Chrun 
prediction = model.predict(scaled_array)
prediction_proba = prediction[0][0]
prediction_proba

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 173ms/step


np.float32(0.1219407)

In [10]:
if prediction_proba > 0.5:
    print('The customer is likely to churn.')
else:
    print('The customer is not likely to churn.')

The customer is not likely to churn.
